In [1]:
import pandas as pd

df = pd.read_csv('train.csv')
df_geo = pd.read_csv('devices.csv')

df = df.merge(df_geo, on='deviceId', how='left')

In [2]:
df.head()

,deviceId,timedate,period,t1,t2,t3,t4,t5,t6,t7,...,t10,t11,t12,t13,x1,x2,x3,deviceType,latitude,longitude
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:00:00,train,0.29,0.05,0.0,0.43,0.47,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:05:00,train,0.29,0.05,0.0,0.39,0.46,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:10:00,train,0.29,0.05,0.0,0.38,0.46,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:15:00,train,0.29,0.05,0.0,0.38,0.45,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:20:00,train,0.29,0.05,0.0,0.37,0.45,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3


In [3]:
import numpy as np

features = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9', 't10', 't11', 't12', 't13', 'x1', 'x3', 'latitude', 'longitude', 'deviceType']

corr_matrix = df[features].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

threshold = 0.90
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

print(f"{to_drop}")

features_filtered = [f for f in features if f not in to_drop]

['t6', 't12', 't13']


In [4]:
df['timedate'] = pd.to_datetime(df['timedate'])

df_clean = df.dropna(subset=features_filtered + ['x2'])

X = df_clean[features_filtered]
y = df_clean['x2']

In [5]:
df_clean.head()
df = None

In [6]:
import pickle

with open('model_5m_flaml.pkl', 'rb') as f:
    automl = pickle.load(f)
    
starting_points = automl.best_config_per_estimator

In [7]:
from flaml import AutoML

# automl = AutoML()

settings = {
    "time_budget": 600,
    "metric": 'mae',
    "task": 'regression',
    "estimator_list": ['xgboost', 'lgbm', "catboost"]
}

automl.fit(X_train=X, y_train=y, **settings, starting_points=starting_points)

[flaml.automl.logger: 03-15 07:18:17] {1752} INFO - task = regression
[flaml.automl.logger: 03-15 07:18:17] {1763} INFO - Evaluation method: holdout
[flaml.automl.logger: 03-15 07:18:45] {1862} INFO - Minimizing error metric: mae
[flaml.automl.logger: 03-15 07:18:45] {1979} INFO - List of ML learners in AutoML Run: ['xgboost', 'lgbm', 'catboost']
[flaml.automl.logger: 03-15 07:18:45] {2282} INFO - iteration 0, current learner xgboost
[flaml.automl.logger: 03-15 07:18:46] {2417} INFO - Estimated sufficient time budget=8289849s. Estimated necessary time budget=9119s.
[flaml.automl.logger: 03-15 07:18:46] {2466} INFO -  at 78.0s,	estimator xgboost's best error=0.1629,	best estimator xgboost's best error=0.1629
[flaml.automl.logger: 03-15 07:18:46] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 03-15 07:18:46] {2466} INFO -  at 78.8s,	estimator lgbm's best error=0.1597,	best estimator lgbm's best error=0.1597
[flaml.automl.logger: 03-15 07:18:47] {2282} INFO - iterat

KeyboardInterrupt: 

In [ ]:
# import pandas as pd
df = None
df_clean = None
df_geo = pd.read_csv('devices.csv')

In [ ]:

df_valid = pd.read_csv('valid.csv')
df_test = pd.read_csv('test.csv')

df_combined = pd.concat([df_valid, df_test], ignore_index=True)
df_combined = df_combined.merge(df_geo, on='deviceId', how='left')

df_combined['timedate'] = pd.to_datetime(df_combined['timedate'])
df_combined['year'] = df_combined['timedate'].dt.year
df_combined['month'] = df_combined['timedate'].dt.month

X_test = df_combined[features_filtered]

predictions = automl.predict(X_test)

df_combined['5min_prediction'] = predictions

submission = df_combined.groupby(['deviceId', 'year', 'month'])['5min_prediction'].mean().reset_index()

submission.rename(columns={'5min_prediction': 'prediction'}, inplace=True)

submission.to_csv('submission.csv', index=False)

[flaml.automl.logger: 03-15 06:49:32] {612} WARNING - No estimator is trained. Please run fit with enough budget.


In [11]:
import pickle

with open('model_5m_flaml.pkl', 'wb') as f:
    pickle.dump(automl, f)